In [1]:
!pip install langchain_groq
# !pip install langfuse==4.14.5

## Prompt Template

In [2]:
from langchain_core.prompts import PromptTemplate


template = """Tell me a joke about {topic}. make sure the joke the {topic} in only"""


prompt = PromptTemplate(
    input_variables=["topic"],
    template=template,
)

In [3]:
prompt.invoke({'topic': 'animal'})

StringPromptValue(text='Tell me a joke about animal. make sure the joke the animal in only')

In [4]:
prompt.format(topic="cricket")

'Tell me a joke about cricket. make sure the joke the cricket in only'

In [5]:
# !pip install langchain_groq

In [6]:
import os
from langchain_groq import ChatGroq


from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

# To create llm object, make sure you have 'GROQ_API_KEY' in your environment variable
# comment above line of code where we are fetching colab userdata.

llm = ChatGroq(temperature=0.9, model_name="openai/gpt-oss-120b")
response = llm.invoke("hi")
response.content

'Hello! How can I help you today?'

In [7]:
input_content = prompt.invoke({'topic': 'cricket'})
print(llm.invoke(input_content).content)

Why did the cricket bring a ladder to the meadow?

Because it heard the grass was *high* and wanted to *reach* the top note! 🎶🐞


In [8]:
llm.invoke(input_content)

AIMessage(content='Why did the cricket player bring a ladder to the game? Because he heard the scores were through the roof!', additional_kwargs={'reasoning_content': 'User wants a joke about cricket, ensuring "the cricket" appears only? They said "make sure the joke the cricket in only". Likely they mean the word "cricket" appears only once? Or that the joke is about cricket only, no other topics. Probably they want a joke that only mentions cricket, not other sports. Provide a short joke about cricket, with "cricket" only mentioned once maybe. Let\'s craft: "Why did the batsman bring a ladder to the match? Because he heard the scores were through the roof!" That doesn\'t mention cricket word. But they asked ensure "the cricket in only". Could be a typo: "make sure the joke the cricket in only" maybe they meant "the cricket is only in the joke"? Or "the cricket is only in the joke"? Probably they want the word "cricket" used only once. So include the word cricket once. Provide a joke 

In [9]:
chain = prompt | llm

print(chain.invoke({"topic": 'cricket'}).content)

Why did the cricket bring a ladder to the pond?

Because it heard the flies were *up* there and wanted to *reach* new heights!


## Chat Prompt Template

In [10]:
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate([
    ("system", "You are a helpful assistant"),
    ("user", "Tell me a joke about {topic}")
])

prompt = prompt_template.invoke({"topic": "kids"})

response = llm.invoke(prompt)

In [11]:
print(response.content)

Sure! Here’s a kid‑friendly joke for you:

**Why did the kid bring a ladder to school?**

Because she wanted to go to high **“school”**! 😄


## Parser

In [12]:
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate

class SentimentParser(BaseModel):
    joke: str = Field(description="one liner joke for the user based on topic.")
    rating: str = Field(description = "return a number between 0 to 5, which is rating of joke as to how funny it was according to you on a scale of 0 to 5, 5 being highest")

# creating llm object with structured output parser
llm_parser = llm.with_structured_output(SentimentParser)

prompt_template = ChatPromptTemplate([
    ("system", "You are a helpful assistant"),
    ("user", "Tell me a joke about {topic}")
])

response = llm_parser.invoke(prompt_template.format_prompt(topic="kids"))

In [13]:
response

SentimentParser(joke='Why did the kid bring a ladder to school? Because he heard the grades were going through the roof!', rating='4')

In [14]:
response.joke

'Why did the kid bring a ladder to school? Because he heard the grades were going through the roof!'

In [15]:
response.rating

'4'